# Short notebook 1 — LP + LightGBM (our best Kaggle submission)

This notebook reproduces our best Kaggle prediction end to end from the raw competition data.

**Idea.** The target is the on/off decision of SINTEF's SHOP optimiser, and every input SHOP used
is in the data, including the SHOP-ready topology file. So instead of learning the input→commitment
map from scratch we

1. **rebuild the weekly optimisation problem SHOP solves** from `Tokke_Vinje_topology.yaml` —
   water balance over 17 reservoirs, rivers and tunnels, turbine/generator efficiency, friction
   losses, start/stop costs, min-flow and min-volume constraints, terminal water value — and solve
   its LP relaxation for every case with HiGHS (~8 s per case on one core on average);
2. turn the solution (commitment, power, flow, local water value from the duals, reservoir
   trajectories) plus a few input features into one row per (case, hour, unit);
3. fit **one LightGBM model** on all 2,922 labelled cases and predict the 725 test cases.

Walk-forward validation (six half-year folds, 2020–2022): **0.9912** micro-AUC, against 0.9619 for
our previous XGBoost model on the same folds. Details and interpretation are in the report.

**Runtime** on a 4-core machine: ~2 h for the 3,647 LPs (cached to `lp_cache/` after the first
run), ~6 min for features and training. Peak memory ~5 GB.

**Requirements:** `numpy pandas pyarrow pyyaml scipy highspy lightgbm joblib`
(developed with highspy 1.15.1, lightgbm 4.7.0, pandas 3.0, Python 3.13).

In [1]:
from __future__ import annotations

import re
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import highspy
import lightgbm as lgb
import numpy as np
import pandas as pd
import yaml
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")

# Folder holding data/kernel, data/extended, prediction_mapping.csv, sample_submission.csv
# (the extracted `sintef_project` folder). Found automatically next to or above this notebook.
CANDIDATES = [Path("."), Path(".."), Path("sintef_project"), Path("../sintef_project")]
ROOT = next(p.resolve() for p in CANDIDATES if (p / "data/kernel").is_dir())
N_JOBS = -1                       # LP solves run in parallel, one HiGHS thread per process
LP_CACHE = Path("lp_cache")       # solved LPs are stored here and reused on a re-run
OUT_PATH = Path("submission_lp_lightgbm.csv")
print("data root:", ROOT)

data root: /home/user/ML_Project/sintef_project


## 1. Topology: the SHOP model of Tokke–Vinje

Parsed from the SHOP-ready YAML: reservoirs (volume→level curves), plants (outlet level, friction
losses), generators (p_min/p_max, turbine efficiency curves at reference heads, generator
efficiency, start/stop costs), rivers (bypass gates, spillways, costed rivers), tunnels, and the
connections between them.

In [2]:
T = 168             # hours per case
K = 0.0036          # Mm3 per (m3/s * hour)
G = 9.81e-3         # MW per (m3/s * m): P = G * H * Q * eta
BIG_PEN = 5_000.0   # EUR per (m3/s*h) of min-flow deficit
# SHOP's tie-break between identical sibling units, measured on the labels (report section 5):
# one Tokke unit on is G1 99% of the time, two are G1+G3 94%; Vinje is always G1, then G1+G2.
SIBLING_ORDER = (("Tokke_G1", "Tokke_G3", "Tokke_G4", "Tokke_G2"),
                 ("Vinje_G1", "Vinje_G2", "Vinje_G3"))


def sanitize(name: str) -> str:
    return name.replace(" ", "_")


@dataclass
class Unit:
    name: str
    plant: str
    penstock: int
    pmin: float
    pmax: float
    gen_x: np.ndarray
    gen_y: np.ndarray
    curves: list            # [(head, q[], eta[])] sorted by head
    start: float
    stop: float


@dataclass
class Plant:
    name: str
    outlet: float
    main_loss: float
    pen_loss: list
    units: list = field(default_factory=list)
    up: list = field(default_factory=list)      # [(type, name)] reservoir or tunnel
    down: tuple | None = None                   # (type, name) or None = sea


@dataclass
class Topology:
    res: dict
    rivers: dict
    plants: dict
    units: dict
    tunnels: dict
    river_up: dict          # river -> (type, name)
    river_down: dict        # river -> (type, name) or None
    tunnel_up: dict
    tunnel_down: dict


def _f(x):
    return float(x[0]) if isinstance(x, list) else float(x)


def load_topology() -> Topology:
    d = yaml.safe_load(open(ROOT / "data/extended/Tokke_Vinje_topology.yaml"))
    m = d["model"]
    res = {k: dict(max_vol=float(v["max_vol"]), lrl=float(v["lrl"]), hrl=float(v["hrl"]),
                   vx=np.array(v["vol_head"]["x"], float), vy=np.array(v["vol_head"]["y"], float))
           for k, v in m["reservoir"].items()}
    plants = {k: Plant(k, float(v["outlet_line"]), _f(v["main_loss"]),
                       [float(x) for x in v["penstock_loss"]]) for k, v in m["plant"].items()}
    units = {}
    for k, v in m["generator"].items():
        curves = sorted(((float(c["ref"]), np.array(c["x"], float), np.array(c["y"], float) / 100)
                         for c in v["turb_eff_curves"]), key=lambda c: c[0])
        units[k] = Unit(k, "", int(v["penstock"]), float(v["p_min"]), float(v["p_max"]),
                        np.array(v["gen_eff_curve"]["x"], float),
                        np.array(v["gen_eff_curve"]["y"], float) / 100, curves,
                        float(v["startcost_const"]), float(v["stopcost_const"]))
    tunnels = {k: dict(loss=float(v["loss_factor"])) for k, v in m["tunnel"].items()}
    rivers = m["river"]
    river_up, river_down, tunnel_up, tunnel_down = {}, {}, {}, {}
    for c in d["connections"]:
        ft, f, tt, to = c["from_type"], c["from"], c["to_type"], c["to"]
        if ft == "generator":
            units[f].plant = to
            plants[to].units.append(f)
        elif tt == "plant":
            plants[to].up.append((ft, f))
            if ft == "tunnel":
                tunnel_down[f] = (tt, to)
        elif ft == "plant":
            plants[f].down = (tt, to)
        elif tt == "river":
            if ft == "reservoir":
                river_up[to] = (ft, f)
            else:                       # river -> river (into r_Vest_Vassdraget)
                river_down[f] = (tt, to)
        elif ft == "river":
            river_down[f] = (tt, to)
        elif tt == "tunnel":
            tunnel_up[to] = (ft, f)
        elif ft == "tunnel":
            tunnel_down[f] = (tt, to)
    for r in rivers:
        river_down.setdefault(r, None)
    return Topology(res, rivers, plants, units, tunnels, river_up, river_down,
                    tunnel_up, tunnel_down)


def level(topo: Topology, r: str, vol: float) -> float:
    R = topo.res[r]
    return float(np.interp(vol, R["vx"], R["vy"]))


def turbine_eta(u: Unit, q: np.ndarray, head: float) -> np.ndarray:
    """Turbine efficiency at flow q, linearly interpolated between the reference-head curves."""
    hs = [c[0] for c in u.curves]
    if head <= hs[0]:
        c = u.curves[0]
        return np.interp(q, c[1], c[2])
    if head >= hs[-1]:
        c = u.curves[-1]
        return np.interp(q, c[1], c[2])
    j = int(np.searchsorted(hs, head)) - 1
    (h0, x0, y0), (h1, x1, y1) = u.curves[j], u.curves[j + 1]
    w = (head - h0) / (h1 - h0)
    return (1 - w) * np.interp(q, x0, y0) + w * np.interp(q, x1, y1)


def q_range(u: Unit) -> tuple[float, float]:
    lo = max(c[1][0] for c in u.curves)
    hi = min(c[1][-1] for c in u.curves)
    return lo, hi


def unit_pq(u: Unit, h_gross: float, plant: Plant, n: int = 40):
    """Single-unit PQ points at gross head h_gross (losses are added separately at plant level).

    Efficiency is evaluated at the head net of this unit's own friction losses, which is the
    head the turbine actually sees when it runs alone.
    """
    qlo, qhi = q_range(u)
    q = np.linspace(qlo, qhi, n)
    kpen = plant.pen_loss[u.penstock - 1] if len(plant.pen_loss) >= u.penstock else 0.0
    h_eval = h_gross - (plant.main_loss + kpen) * q ** 2
    eta_t = turbine_eta(u, q, h_eval.mean())
    p = G * h_gross * q * eta_t
    p = p * np.interp(p, u.gen_x, u.gen_y)
    # operating range is capped at p_max; keep at least two points
    keep = p <= u.pmax + 1e-9
    keep[:2] = True
    return q[keep], p[keep]


def concave_envelope(q: np.ndarray, p: np.ndarray):
    """Upper concave hull of (q, p) starting at the first point. Returns breakpoints."""
    pts = []
    for x, y in zip(q, p):
        while len(pts) >= 2:
            (x1, y1), (x2, y2) = pts[-2], pts[-1]
            # remove middle point if it lies below the chord
            if (y2 - y1) * (x - x1) <= (y - y1) * (x2 - x1):
                pts.pop()
            else:
                break
        pts.append((x, y))
    a = np.array(pts)
    return a[:, 0], a[:, 1]


def head_for_plant(topo: Topology, plant: Plant, V0: dict) -> float:
    """Gross head from the initial upstream level to max(outlet line, downstream level)."""
    ups = []
    for t, n in plant.up:
        if t == "reservoir":
            ups.append(level(topo, n, V0[n]))
        elif t == "tunnel":
            r = topo.tunnel_up[n][1]
            ups.append(level(topo, r, V0[r]))
    up = float(np.mean(ups))
    tail = plant.outlet
    if plant.down and plant.down[0] == "reservoir":
        tail = max(tail, level(topo, plant.down[1], V0[plant.down[1]]))
    return max(up - tail, 1.0)


def plant_conversion(topo: Topology, plant: Plant, head: float) -> float:
    """MWh per Mm3 at the best operating point of the plant's first unit."""
    u = topo.units[plant.units[0]]
    q, p = unit_pq(u, head, plant)
    return float(np.max(p / q)) / K


def main_path(topo: Topology, r: str) -> list[tuple[str, str]]:
    """(plant, feeding reservoir) for every plant on the main waterway from r to the sea.

    The feeding reservoir is the last reservoir the water passed through before the plant.
    """
    out, seen, node, last = [], set(), ("reservoir", r), r
    while node and node not in seen:
        seen.add(node)
        t, n = node
        if t == "reservoir":
            last = n
        if t == "plant":
            out.append((n, last))
            node = topo.plants[n].down
            continue
        if t == "river":
            node = topo.river_down.get(n)
            continue
        # reservoir: plant > tunnel > main river
        nxt = None
        for p in topo.plants.values():
            if ("reservoir", n) in p.up:
                nxt = ("plant", p.name)
        if nxt is None:
            for tn, (ut, un) in topo.tunnel_up.items():
                if un == n:
                    dt, dn = topo.tunnel_down[tn]
                    nxt = (dt, dn)
        if nxt is None:
            for rv, (ut, un) in topo.river_up.items():
                if un == n and topo.rivers[rv].get("main_river"):
                    nxt = topo.river_down[rv]
        node = nxt
    return out


def unit_up_down(topo: Topology, unit: str) -> tuple[str, str | None]:
    """Reservoir a unit draws from, and the reservoir its discharge reaches first (or None)."""
    P = topo.plants[topo.units[unit].plant]
    t, n = P.up[0]
    up = n if t == "reservoir" else topo.tunnel_up[n][1]
    node = P.down
    while node and node[0] == "river":
        node = topo.river_down.get(node[1])
    return up, (node[1] if node else None)


TOPO = load_topology()
print(f"{len(TOPO.res)} reservoirs, {len(TOPO.plants)} plants, {len(TOPO.units)} units, "
      f"{len(TOPO.rivers)} rivers, {len(TOPO.tunnels)} tunnels")

17 reservoirs, 9 plants, 14 units, 37 rivers, 5 tunnels


## 2. Inputs per case

A case starts at 00:00 UTC on its start date and covers 168 hours. Its inputs are the hourly
price and inflows, the reservoir volumes at the start, the water value at start + 7 days (the
terminal water value, as the dataset documentation specifies), and the min-flow / min-volume
series.

In [3]:
def load_raw():
    k = ROOT / "data/kernel"
    e = ROOT / "data/extended"

    def rd(p):
        df = pd.read_csv(p)
        df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
        return df.set_index("date")
    price = rd(k / "Historical_day_ahead_price_2015_2025.csv").iloc[:, 0]
    inflow = rd(k / "Historical_inflow_1958_2025.csv")
    inflow = inflow[inflow.index >= "2014-12-01"]
    vol = rd(k / "Historical_volume_2015_2024.csv")
    wv = rd(k / "Synthetic_water_value_2015_2024.csv")
    minvol = rd(e / "Constraint_min_volume.csv")
    minflow = rd(e / "Constraint_min_flow.csv")
    return price, inflow, vol, wv, minvol, minflow


def case_inputs(start: pd.Timestamp) -> dict:
    price, inflow, vol, wv, minvol, minflow = RAW
    idx = pd.date_range(start, periods=T, freq="h")
    end = start + pd.Timedelta(days=7)
    return dict(
        price=price.reindex(idx).ffill().bfill().to_numpy(float),
        inflow=inflow.reindex(idx).ffill().bfill(),
        V0=vol.loc[start].to_dict(),
        wv=wv.loc[end].to_dict() if end in wv.index else wv.iloc[-1].to_dict(),
        minvol=minvol.reindex(idx).ffill().fillna(0.0),
        minflow=minflow.reindex(idx).ffill().fillna(0.0),
    )


def all_cases() -> pd.DataFrame:
    uc = pd.read_csv(ROOT / "data/kernel/Unit_commitment_decisions.csv",
                     usecols=["Run No", "starttime"])
    pm = pd.read_csv(ROOT / "prediction_mapping.csv")
    c = pd.concat([uc, pm], ignore_index=True).rename(columns={"Run No": "run_no"})
    c["starttime"] = pd.to_datetime(c["starttime"])
    return c


RAW = load_raw()
CASES = all_cases()
print(f"{len(CASES)} cases: {(CASES.starttime.dt.year < 2023).sum()} labelled, "
      f"{(CASES.starttime.dt.year >= 2023).sum()} test")

3647 cases: 2922 labelled, 725 test


## 3. The weekly optimisation problem

One linear programme per case (the LP relaxation of SHOP's unit-commitment problem):

* **objective** — maximise market revenue minus friction losses, start/stop costs, river,
  bypass and spill costs and constraint penalties, plus the value of the water left at the end;
* **units** — on/off variable $u \in [0,1]$ carrying the minimum block ($q_{\min}$, $p_{\min}$),
  plus segments of the concave envelope of the PQ curve at the case's initial head;
* **water** — hourly balance in every reservoir, the Vest-Vassdraget river junction and the
  Songa tunnel junction; reservoir bounds; min volume and min flow as heavily penalised soft
  constraints;
* **terminal value** — every plant's break-even price equals the water value of the reservoir
  feeding it (measured from the labels, report section 5).

The LP's duals give the hourly shadow price of water in every reservoir, from which each unit's
*local water value* (EUR/MWh) and margin (price − local water value) follow.

In [4]:
class _Builder:
    def __init__(self):
        self.lb, self.ub, self.c = [], [], []
        self.rows_lo, self.rows_hi = [], []
        self.ri, self.ci, self.v = [], [], []
        self.n = 0
        self.m = 0

    def var(self, shape, lb=0.0, ub=np.inf, cost=0.0):
        size = int(np.prod(shape))
        idx = np.arange(self.n, self.n + size).reshape(shape)
        self.n += size
        self.lb.append(np.broadcast_to(np.asarray(lb, float), shape).ravel())
        self.ub.append(np.broadcast_to(np.asarray(ub, float), shape).ravel())
        self.c.append(np.broadcast_to(np.asarray(cost, float), shape).ravel())
        return idx

    def rows(self, terms, lo, hi):
        """terms: list of (col_idx[T], coef scalar or [T]); one row per element."""
        nrow = len(terms[0][0])
        r = np.arange(self.m, self.m + nrow)
        self.m += nrow
        for idx, coef in terms:
            idx = np.asarray(idx)
            self.ri.append(r)
            self.ci.append(idx)
            self.v.append(np.broadcast_to(np.asarray(coef, float), idx.shape).astype(float))
        self.rows_lo.append(np.broadcast_to(np.asarray(lo, float), (nrow,)))
        self.rows_hi.append(np.broadcast_to(np.asarray(hi, float), (nrow,)))
        return r

    def solve(self, threads: int = 1):
        """Solve the LP with HiGHS (dual simplex); returns primal, duals, status."""
        import scipy.sparse as sp
        lb, ub, c = (np.concatenate(x) for x in (self.lb, self.ub, self.c))
        A = sp.csc_matrix((np.concatenate(self.v), (np.concatenate(self.ri),
                                                    np.concatenate(self.ci))),
                          shape=(self.m, self.n))
        lp = highspy.HighsLp()
        lp.num_col_, lp.num_row_ = self.n, self.m
        lp.col_cost_, lp.col_lower_, lp.col_upper_ = c, lb, ub
        lp.row_lower_ = np.concatenate(self.rows_lo)
        lp.row_upper_ = np.concatenate(self.rows_hi)
        lp.a_matrix_.format_ = highspy.MatrixFormat.kColwise
        lp.a_matrix_.start_ = A.indptr
        lp.a_matrix_.index_ = A.indices
        lp.a_matrix_.value_ = A.data
        lp.sense_ = highspy.ObjSense.kMaximize
        h = highspy.Highs()
        h.setOptionValue("output_flag", False)
        h.setOptionValue("threads", threads)
        h.passModel(lp)
        h.run()
        sol = h.getSolution()
        status = h.modelStatusToString(h.getModelStatus())
        x = np.array(sol.col_value)
        y = np.array(sol.row_dual) if sol.dual_valid else None
        return x, y, status


def solve_case(topo: Topology, inp: dict) -> dict:
    """Build and solve one weekly case. Returns arrays keyed by unit / reservoir."""
    price = inp["price"]
    V0 = {r: float(inp["V0"][r]) for r in topo.res}
    B = _Builder()

    # --- heads, conversion factors, terminal values ---
    head = {p: head_for_plant(topo, P, V0) for p, P in topo.plants.items()}
    conv = {p: plant_conversion(topo, P, head[p]) for p, P in topo.plants.items()}
    # terminal value, EUR/Mm3: at every plant k on the way to the sea, conv_k MWh priced at
    # the water value of the reservoir feeding k (measured from the labels, see the report)
    wvm = {r: sum(inp["wv"][f] * conv[pl] for pl, f in main_path(topo, r)) for r in topo.res}

    # --- units ---
    U = {}
    for name, u in topo.units.items():
        P = topo.plants[u.plant]
        q, p = unit_pq(u, head[u.plant], P)
        qe, pe = concave_envelope(q, p)
        w = np.diff(qe)
        slope = np.diff(pe) / w
        # u carries the minimum block (qmin -> pmin); segments carry the rest of the PQ curve
        uu = B.var((T,), 0, 1, price * pe[0])
        st = B.var((T,), 0, 1, -u.start)
        sd = B.var((T,), 0, 1, -u.stop)
        seg = B.var((len(w), T), 0, np.repeat(w[:, None], T, 1), slope[:, None] * price[None, :])
        U[name] = dict(u=uu, st=st, sd=sd, seg=seg, qmin=qe[0], pmin=pe[0], w=w, slope=slope)
        for k in range(len(w)):
            B.rows([(seg[k], 1.0), (uu, -w[k])], -np.inf, 0.0)
        B.rows([(st[1:], 1.0), (uu[1:], -1.0), (uu[:-1], 1.0)], 0.0, np.inf)
        B.rows([(sd[1:], 1.0), (uu[1:], 1.0), (uu[:-1], -1.0)], 0.0, np.inf)

    # SHOP's tie-break between identical siblings, as measured on the labels: one Tokke unit
    # on is G1 99% of the time, two are G1+G3 94%, three G1+G3+G4 66%; Vinje is G1, G1+G2.
    # Encoding the order turns "which unit" into "how many units" and breaks MIP symmetry.
    for chain in SIBLING_ORDER:
        for a, b in zip(chain[:-1], chain[1:]):
            B.rows([(U[a]["u"], 1.0), (U[b]["u"], -1.0)], 0.0, np.inf)

    def flow_terms(unit_names, sign=1.0):
        """Terms whose sum is the total discharge of the listed units."""
        out = []
        for nm in unit_names:
            d = U[nm]
            out.append((d["u"], sign * d["qmin"]))
            for k in range(len(d["w"])):
                out.append((d["seg"][k], sign))
        return out

    # --- friction losses: L >= tangent cuts of c*Q^3, priced at max(price, 0) ---
    ppos = np.maximum(price, 0.0)
    for pn, P in topo.plants.items():
        groups = []
        if P.main_loss > 0:
            groups.append((P.main_loss, P.units))
        for j, kp in enumerate(P.pen_loss):
            on_pen = [nm for nm in P.units if topo.units[nm].penstock == j + 1]
            if kp > 0 and on_pen:
                groups.append((kp, on_pen))
        eta = 0.9
        for kloss, members in groups:
            qmax = sum(q_range(topo.units[nm])[1] for nm in members)
            c3 = G * eta * kloss          # MW lost = c3 * Q^3
            L = B.var((T,), 0, np.inf, -ppos)
            for qj in np.linspace(qmax / 8, qmax, 8):
                a, b = 3 * c3 * qj ** 2, -2 * c3 * qj ** 3
                B.rows([(L, 1.0)] + flow_terms(members, -a), b, np.inf)

    # --- rivers ---
    F = {}
    mf = inp["minflow"]
    for rv, R in topo.rivers.items():
        if rv == "r_Vest_Vassdraget":
            F[rv] = B.var((T,), 0, np.inf, 0)
            continue
        if "flow_cost_curve" in R:
            x = np.array(R["flow_cost_curve"]["x"], float)
            y = np.array(R["flow_cost_curve"]["y"], float)
            # y is the cost per hour at flow x, so the marginal cost is the slope, made
            # non-decreasing so the LP fills segments in order (measured on Kjela, see report)
            mc = np.maximum.accumulate(np.diff(y) / np.diff(x))
            F[rv] = [B.var((T,), 0, x[j + 1] - x[j], -mc[j]) for j in range(len(x) - 1)]
        else:
            cost = float(R.get("flow_cost_const", 0.0))
            ub = float(R.get("max_flow_const", np.inf)) if rv.startswith("b_") else np.inf
            F[rv] = B.var((T,), 0, ub, -cost)
        if rv in mf.columns:
            req = mf[rv].to_numpy(float)
            if req.max() > 0:
                sl = B.var((T,), 0, np.inf, -BIG_PEN)
                terms = [(s, 1.0) for s in F[rv]] if isinstance(F[rv], list) else [(F[rv], 1.0)]
                B.rows(terms + [(sl, 1.0)], req, np.inf)

    def river_terms(rv, sign):
        f = F[rv]
        return [(s, sign) for s in f] if isinstance(f, list) else [(f, sign)]

    # --- tunnels ---
    TN = {}
    for tn in topo.tunnels:
        TN[tn] = B.var((T,), -1000.0, 1000.0, 0.0)

    # --- reservoirs ---
    V = {}
    mv = inp["minvol"]
    for r, R in topo.res.items():
        ub = max(R["max_vol"], 0.0)
        cost = np.zeros(T)
        cost[-1] = wvm[r]
        V[r] = B.var((T,), 0.0, ub, cost)
        if r in mv.columns:
            req = mv[r].to_numpy(float)
            if req.max() > 0:
                sl = B.var((T,), 0, np.inf, -1e5)
                B.rows([(V[r], 1.0), (sl, 1.0)], req, np.inf)

    # water balance: V[t] - V[t-1] - K*(in - out) = K*inflow  (V[-1] = V0)
    infl = inp["inflow"]
    bal_rows = {}
    for r in topo.res:
        ins, outs = [], []
        for pn, P in topo.plants.items():
            if P.down == ("reservoir", r):
                ins += flow_terms(P.units, 1.0)
            if ("reservoir", r) in P.up:
                outs += flow_terms(P.units, 1.0)
        for rv in topo.rivers:
            if topo.river_down.get(rv) == ("reservoir", r):
                ins += river_terms(rv, 1.0)
            if topo.river_up.get(rv) == ("reservoir", r):
                outs += river_terms(rv, 1.0)
        for tn in topo.tunnels:
            if topo.tunnel_down.get(tn) == ("reservoir", r):
                ins.append((TN[tn], 1.0))
            if topo.tunnel_up.get(tn) == ("reservoir", r):
                outs.append((TN[tn], 1.0))
        rhs = K * infl[r].to_numpy(float) if r in infl.columns else np.zeros(T)
        rhs = rhs.copy()
        rhs[0] += V0[r]
        # build the per-t rows: V[t] - V[t-1] - K*sum(ins) + K*sum(outs)
        rows_terms = [(V[r], 1.0)]
        rows_terms += [(i, -K * np.asarray(c)) for i, c in ins]
        rows_terms += [(i, K * np.asarray(c)) for i, c in outs]
        rid = B.rows(rows_terms, rhs, rhs)
        # add -V[t-1] for t >= 1 directly into the COO
        B.ri.append(rid[1:])
        B.ci.append(V[r][:-1])
        B.v.append(-np.ones(T - 1))
        bal_rows[r] = rid

    # river junction r_Vest_Vassdraget: F = own inflow + inputs; F flows into Bandak (above)
    ins = []
    for rv in topo.rivers:
        if topo.river_down.get(rv) == ("river", "r_Vest_Vassdraget"):
            ins += river_terms(rv, -1.0)
    for pn, P in topo.plants.items():
        if P.down == ("river", "r_Vest_Vassdraget"):
            ins += flow_terms(P.units, -1.0)
    rhs = infl["r_Vest_Vassdraget"].to_numpy(float)
    B.rows([(F["r_Vest_Vassdraget"], 1.0)] + ins, rhs, rhs)
    if "r_Vest_Vassdraget" in mf.columns:
        req = mf["r_Vest_Vassdraget"].to_numpy(float)
        sl = B.var((T,), 0, np.inf, -BIG_PEN)
        B.rows([(F["r_Vest_Vassdraget"], 1.0), (sl, 1.0)], req, np.inf)

    # plants fed by tunnels (Songa): plant flow = sum of tunnel flows
    for pn, P in topo.plants.items():
        tins = [n for t, n in P.up if t == "tunnel"]
        if tins:
            B.rows(flow_terms(P.units, 1.0) + [(TN[t], -1.0) for t in tins], 0.0, 0.0)

    x, y, status = B.solve()

    out = dict(status=status)
    if x.size == 0 or not np.isfinite(x).all():
        return out
    for nm, d in U.items():
        u = x[d["u"]]
        q = d["qmin"] * u + sum(x[d["seg"][k]] for k in range(len(d["w"])))
        p = d["pmin"] * u + sum(d["slope"][k] * x[d["seg"][k]] for k in range(len(d["w"])))
        out[f"u_{nm}"] = u
        out[f"q_{nm}"] = q
        out[f"p_{nm}"] = p
    for rv, f in F.items():
        out[f"f_{rv}"] = sum(x[s] for s in f) if isinstance(f, list) else x[f]
    for r in topo.res:
        out[f"v_{r}"] = x[V[r]]
        if y is not None:
            # dual of the balance row = EUR per Mm3 of extra water in r at hour t
            out[f"mu_{r}"] = np.abs(y[bal_rows[r]])
    if y is not None:
        # local water value of each unit, EUR/MWh: the water it uses is worth mu_up and lands
        # downstream worth mu_down; divide by MWh per Mm3 at the unit's best point
        for nm, d in U.items():
            up, dn = unit_up_down(topo, nm)
            mu_dn = out[f"mu_{dn}"] if dn else 0.0
            best = float(np.max((d["pmin"] + np.r_[0, np.cumsum(d["slope"] * d["w"])])
                                / (d["qmin"] + np.r_[0, np.cumsum(d["w"])])))
            out[f"lwv_{nm}"] = (out[f"mu_{up}"] - mu_dn) * K / best
    return out


## 4. Solve every case

Cases are independent, so they are solved in parallel. Results are stored in float32 tables:
per (case, hour, unit) the commitment `u`, power `p`, flow `q` and local water value `lwv`; per
(case, hour) the reservoir volumes, water shadow prices and river flows. Delete `lp_cache/` to
force a re-solve.

In [5]:
def _tables(run_no: int, o: dict):
    units = list(TOPO.units)
    unit = pd.DataFrame({
        "run_no": np.int32(run_no),
        "hour": np.tile(np.arange(T, dtype=np.int16), len(units)),
        "unit": np.repeat([sanitize(u) for u in units], T),
        **{k: np.concatenate([o[f"{k}_{u}"] for u in units]).astype(np.float32)
           for k in ("u", "p", "q", "lwv") if f"{k}_{units[0]}" in o},
    })
    case = pd.DataFrame({"run_no": np.int32(run_no), "hour": np.arange(T, dtype=np.int16),
                         **{k: np.asarray(v, np.float32) for k, v in o.items()
                            if k.startswith(("v_", "mu_", "f_")) and np.ndim(v) == 1}})
    case["status_ok"] = o["status"] == "Optimal"
    return unit, case


def solve_one(run_no, inp):
    o = solve_case(TOPO, inp)
    assert o["status"] == "Optimal", (run_no, o["status"])
    return _tables(run_no, o)


t0 = time.time()
if (LP_CACHE / "lp_unit.parquet").exists():
    LP_UNIT = pd.read_parquet(LP_CACHE / "lp_unit.parquet")
    LP_CASE = pd.read_parquet(LP_CACHE / "lp_case.parquet")
    print(f"loaded cached LP solutions for {LP_UNIT.run_no.nunique()} cases")
else:
    res = Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(solve_one)(r, case_inputs(s)) for r, s in zip(CASES.run_no, CASES.starttime))
    LP_UNIT = pd.concat([a for a, _ in res], ignore_index=True)
    LP_CASE = pd.concat([b for _, b in res], ignore_index=True)
    LP_CACHE.mkdir(exist_ok=True)
    LP_UNIT.to_parquet(LP_CACHE / "lp_unit.parquet", index=False)
    LP_CASE.to_parquet(LP_CACHE / "lp_case.parquet", index=False)
    print(f"solved {len(res)} LPs in {time.time() - t0:.0f}s")

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.


[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   36.9s


[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:  3.2min


[Parallel(n_jobs=-1)]: Done 154 tasks      | elapsed:  6.5min


[Parallel(n_jobs=-1)]: Done 280 tasks      | elapsed: 11.1min


...

[Parallel(n_jobs=-1)]: Done 2584 tasks      | elapsed: 98.7min


[Parallel(n_jobs=-1)]: Done 3034 tasks      | elapsed: 107.9min


[Parallel(n_jobs=-1)]: Done 3520 tasks      | elapsed: 116.9min


[Parallel(n_jobs=-1)]: Done 3647 out of 3647 | elapsed: 120.1min finished


solved 3647 LPs in 7215s


## 5. Features: one row per (case, hour, unit)

61 features in five groups:

* `lp_` — the optimiser's solution and its temporal context: commitment, power, flow, local water
  value and margin, rolling windows, run lengths, plant-level count of units on, reservoir
  trajectories;
* `ec_` — economics from the inputs: price level/rank/z-score within the week, spread to the
  terminal water value, season, inflow;
* `cn_` — min-flow and min-volume constraint series;
* `st_` — the observed storage trajectory over the week (available for test cases too);
* `s_` — static unit attributes and the unit/plant identity.

In [6]:
def _roll(x: np.ndarray, w: int) -> np.ndarray:
    """Centred rolling mean along the last axis, edges use the available window."""
    c = np.cumsum(np.pad(x, [(0, 0)] * (x.ndim - 1) + [(1, 0)]), axis=-1, dtype=np.float64)
    n = x.shape[-1]
    h = w // 2
    lo = np.clip(np.arange(n) - h, 0, n)
    hi = np.clip(np.arange(n) + h + 1, 0, n)
    return ((c[..., hi] - c[..., lo]) / (hi - lo)).astype(np.float32)


def _shift(x: np.ndarray, k: int) -> np.ndarray:
    out = np.full_like(x, np.nan)
    if k > 0:
        out[..., k:] = x[..., :-k]
    else:
        out[..., :k] = x[..., -k:]
    return out


def _rank(x: np.ndarray) -> np.ndarray:
    """Rank within the last axis, scaled to [0, 1]."""
    return (np.argsort(np.argsort(x, axis=-1), axis=-1) / (x.shape[-1] - 1)).astype(np.float32)


def _run_features(u: np.ndarray):
    """Length of the current run of the (rounded) state and position inside it."""
    b = (u > 0.5)
    n = b.shape[-1]
    start = np.zeros(b.shape, np.int32)
    for t in range(1, n):
        same = b[..., t] == b[..., t - 1]
        start[..., t] = np.where(same, start[..., t - 1], t)
    end = np.zeros(b.shape, np.int32)
    end[..., -1] = n - 1
    for t in range(n - 2, -1, -1):
        same = b[..., t] == b[..., t + 1]
        end[..., t] = np.where(same, end[..., t + 1], t)
    t_idx = np.arange(n)
    length = (end - start + 1).astype(np.float32)
    since = (t_idx - start).astype(np.float32)
    until = (end - t_idx).astype(np.float32)
    return length, since, until


def build_features(cases: pd.DataFrame, lp_unit: pd.DataFrame, lp_case: pd.DataFrame,
          labels: pd.DataFrame | None = None, prefix: str = "lp"):
    """cases: run_no, starttime. Returns (meta frame, float32 matrix, feature names).

    Rows are in (case, unit, hour) order; `meta` holds run_no, unit index, hour and y.
    """
    topo = TOPO
    units = list(topo.units)
    su = [sanitize(u) for u in units]
    N, U = len(cases), len(units)
    runs = cases["run_no"].to_numpy()
    starts = pd.to_datetime(cases["starttime"]).to_numpy()
    price_s, inflow, vol, wv, minvol, minflow = RAW

    # ---- optimiser tensors (case, unit, hour)
    lu = lp_unit.set_index(["run_no", "unit", "hour"]).sort_index()
    idx = pd.MultiIndex.from_product([runs, su, range(T)], names=["run_no", "unit", "hour"])
    lu = lu.reindex(idx)

    def ten(col):
        return lu[col].to_numpy(np.float32).reshape(N, U, T)
    u, p, q = ten("u"), ten("p"), ten("q")
    lwv = ten("lwv") if "lwv" in lu.columns else None
    lc = lp_case.set_index(["run_no", "hour"]).sort_index()
    lc = lc.reindex(pd.MultiIndex.from_product([runs, range(T)]))

    # ---- inputs per case
    hours = (starts[:, None] + np.arange(T)[None, :] * np.timedelta64(1, "h"))
    price = price_s.reindex(hours.ravel()).ffill().to_numpy(np.float32).reshape(N, T)
    end = starts + np.timedelta64(7, "D")
    wv_end = wv.reindex(end).ffill()
    wv_0 = wv.reindex(starts).ffill()
    v_0 = vol.reindex(starts).ffill()

    pmax = np.array([topo.units[x].pmax for x in units], np.float32)
    pmin = np.array([topo.units[x].pmin for x in units], np.float32)
    qmax = np.array([q_range(topo.units[x])[1] for x in units], np.float32)
    updn = [unit_up_down(topo, x) for x in units]
    plant_of = [topo.units[x].plant for x in units]
    plants = sorted(set(plant_of))
    merit = {}
    for chain in SIBLING_ORDER:
        for k, nm in enumerate(chain):
            merit[nm] = k
    nsib = np.array([sum(1 for y in units if topo.units[y].plant == topo.units[x].plant)
                     for x in units], np.float32)

    F = {}
    bc = lambda a: np.broadcast_to(np.asarray(a, np.float32), (N, U, T))  # noqa: E731
    pre = f"{prefix}_"

    # ---- lp_: optimiser solution + temporal context
    F[pre + "u"] = u
    F[pre + "p_rel"] = p / pmax[None, :, None]
    F[pre + "q_rel"] = q / qmax[None, :, None]
    for w in (3, 7, 25, 73):
        F[pre + f"u_roll{w}"] = _roll(u, w)
    F[pre + "u_lag1"] = _shift(u, 1)
    F[pre + "u_lead1"] = _shift(u, -1)
    F[pre + "u_week"] = bc(u.mean(-1, keepdims=True))
    F[pre + "p_week"] = bc(F[pre + "p_rel"].mean(-1, keepdims=True))
    day = np.arange(T) // 24
    dmean = np.stack([u[..., day == d].mean(-1) for d in range(7)], -1)
    F[pre + "u_day"] = dmean[..., day]
    length, since, until = _run_features(u)
    F[pre + "run_len"], F[pre + "run_since"], F[pre + "run_until"] = length, since, until
    pl_n = np.zeros_like(u)
    for pn in plants:
        m = np.array([pl == pn for pl in plant_of])
        pl_n[:, m, :] = u[:, m, :].sum(1, keepdims=True)
    F[pre + "plant_n_on"] = pl_n
    if lwv is not None:
        marg = price[:, None, :] - lwv
        F[pre + "lwv"] = lwv
        F[pre + "margin"] = marg
        F[pre + "margin_rel"] = marg / np.maximum(np.abs(lwv), 1.0)
        F[pre + "margin_rank"] = _rank(marg)
        F[pre + "margin_roll7"] = _roll(marg, 7)
        F[pre + "margin_roll25"] = _roll(marg, 25)
        F[pre + "lwv_week"] = bc(lwv.mean(-1, keepdims=True))

    # reservoir state from the optimiser, for the unit's own up/down reservoirs
    vu = np.zeros((N, U, T), np.float32)
    vd = np.zeros((N, U, T), np.float32)
    for j, (r_up, r_dn) in enumerate(updn):
        vu[:, j] = lc[f"v_{r_up}"].to_numpy(np.float32).reshape(N, T) / topo.res[r_up]["max_vol"]
        if r_dn:
            vd[:, j] = lc[f"v_{r_dn}"].to_numpy(np.float32).reshape(N, T) / topo.res[r_dn]["max_vol"]
    F[pre + "vup"] = vu
    F[pre + "vup_end"] = bc(vu[..., -1:])
    F[pre + "vup_min"] = bc(vu.min(-1, keepdims=True))
    F[pre + "vdn"] = vd
    F[pre + "vdn_max"] = bc(vd.max(-1, keepdims=True))

    # ---- ec_: economics from the inputs
    F["ec_price"] = bc(price[:, None, :])
    F["ec_price_rank"] = bc(_rank(price)[:, None, :])
    pz = (price - price.mean(-1, keepdims=True)) / (price.std(-1, keepdims=True) + 1e-3)
    F["ec_price_z"] = bc(pz[:, None, :])
    wvu_end = np.stack([wv_end[r].to_numpy(np.float32) for r, _ in updn], 1)
    wvu_0 = np.stack([wv_0[r].to_numpy(np.float32) for r, _ in updn], 1)
    wvd_end = np.stack([wv_end[r].to_numpy(np.float32) if r else np.zeros(N, np.float32)
                        for _, r in updn], 1)
    F["ec_wv_up_end"] = bc(wvu_end[..., None])
    F["ec_wv_up_change"] = bc((wvu_end - wvu_0)[..., None])
    F["ec_wv_dn_end"] = bc(wvd_end[..., None])
    F["ec_spread"] = price[:, None, :] - wvu_end[..., None]
    F["ec_spread_rel"] = F["ec_spread"] / np.maximum(wvu_end[..., None], 1.0)
    F["ec_hours_above"] = bc((price[:, None, :] > wvu_end[..., None]).mean(-1, keepdims=True))
    hod = (pd.DatetimeIndex(hours.ravel()).hour.to_numpy().reshape(N, T)).astype(np.float32)
    F["ec_hour_of_day"] = bc(hod[:, None, :])
    F["ec_t"] = bc(np.arange(T, dtype=np.float32)[None, None, :])
    doy = pd.DatetimeIndex(starts).dayofyear.to_numpy().astype(np.float32)
    F["ec_doy_sin"] = bc(np.sin(2 * np.pi * doy / 365.25)[:, None, None])
    F["ec_doy_cos"] = bc(np.cos(2 * np.pi * doy / 365.25)[:, None, None])
    v0u = np.stack([v_0[r].to_numpy(np.float32) / topo.res[r]["max_vol"] for r, _ in updn], 1)
    F["ec_v0_up"] = bc(v0u[..., None])
    inf_w = inflow.reindex(hours.ravel()).ffill()
    infu = np.stack([inf_w[r].to_numpy(np.float32).reshape(N, T).mean(-1) for r, _ in updn], 1)
    F["ec_inflow_up_rel"] = bc((infu / qmax[None, :])[..., None])

    # ---- cn_: constraint series shared across units
    mf = minflow.reindex(hours.ravel()).ffill().fillna(0.0)
    for c in mf.columns:
        F[f"cn_minflow_{c}"] = bc(mf[c].to_numpy(np.float32).reshape(N, 1, T))
    mv = minvol.reindex(hours.ravel()).ffill().fillna(0.0)
    for c in mv.columns:
        F[f"cn_minvol_{c}"] = bc((mv[c].to_numpy(np.float32).reshape(N, 1, T)
                                  / topo.res[c]["max_vol"]))

    # ---- st_: observed storage trajectory (post-decision proxy for the week's release)
    days = starts[:, None] + np.arange(8)[None, :] * np.timedelta64(1, "D")
    vd_obs = vol.reindex(days.ravel()).ffill()
    inf_d = inflow.resample("D").mean()
    inf_days = inf_d.reindex(days[:, :7].ravel()).ffill()
    rel_day = np.zeros((N, U, T), np.float32)
    dv_up = np.zeros((N, U), np.float32)
    dv_dn = np.zeros((N, U), np.float32)
    for j, (r_up, r_dn) in enumerate(updn):
        vv = vd_obs[r_up].to_numpy(np.float32).reshape(N, 8)
        ii = inf_days[r_up].to_numpy(np.float32).reshape(N, 7) if r_up in inf_days else 0.0
        # net outflow of the upstream reservoir that day, m3/s, relative to plant capacity
        out_d = (vv[:, :-1] - vv[:, 1:]) / (K * 24) + ii
        nsib_j = nsib[j]
        rel_day[:, j] = (out_d / (qmax[j] * nsib_j))[:, day]
        dv_up[:, j] = (vv[:, -1] - vv[:, 0]) / topo.res[r_up]["max_vol"]
        if r_dn:
            vdn = vd_obs[r_dn].to_numpy(np.float32).reshape(N, 8)
            dv_dn[:, j] = (vdn[:, -1] - vdn[:, 0]) / topo.res[r_dn]["max_vol"]
    F["st_release_day"] = rel_day
    F["st_release_week"] = bc(rel_day.mean(-1, keepdims=True))
    F["st_dv_up"] = bc(dv_up[..., None])
    F["st_dv_dn"] = bc(dv_dn[..., None])

    # ---- s_: static
    F["s_unit"] = bc(np.arange(U, dtype=np.float32)[None, :, None])
    F["s_plant"] = bc(np.array([plants.index(x) for x in plant_of], np.float32)[None, :, None])
    F["s_pmax"] = bc(pmax[None, :, None])
    F["s_pmin_frac"] = bc((pmin / pmax)[None, :, None])
    F["s_merit"] = bc(np.array([merit.get(x, 0) for x in units], np.float32)[None, :, None])

    names = list(F)
    meta = pd.DataFrame({
        "run_no": np.repeat(runs, U * T).astype(np.int32),
        "unit": np.tile(np.repeat(np.arange(U, dtype=np.int16), T), N),
        "hour": np.tile(np.arange(T, dtype=np.int16), N * U),
    })
    X = np.empty((N * U * T, len(names)), np.float32)
    for k, nm in enumerate(names):
        X[:, k] = np.asarray(F[nm], np.float32).reshape(-1)
        F[nm] = None
    if labels is not None:
        lab = labels.set_index("starttime") if "starttime" in labels.columns else labels
        y = np.full((N, U, T), np.nan, np.float32)
        have = pd.DatetimeIndex(starts).isin(lab.index)
        sub = lab.loc[pd.DatetimeIndex(starts)[have]]
        for j, nm in enumerate(units):
            y[have, j] = sub[[f"result_committed_{nm}_t{h}" for h in range(T)]].to_numpy(np.float32)
        meta["y"] = y.reshape(-1)
    return meta, X, names


labels = pd.read_csv(ROOT / "data/kernel/Unit_commitment_decisions.csv", parse_dates=["starttime"])
t0 = time.time()
META, X, NAMES = build_features(CASES, LP_UNIT, LP_CASE, labels=labels)
print(f"feature matrix {X.shape} ({X.nbytes / 1e9:.1f} GB) in {time.time() - t0:.0f}s")

feature matrix (8577744, 61) (2.1 GB) in 39s


## 6. LightGBM on all labelled cases, then predict the test cases

Hyperparameters as validated in walk-forward: 600 rounds, learning rate 0.05, 127 leaves,
`min_data_in_leaf=500` (consecutive hours of one unit-week are nearly identical, so the effective
sample size is far below the row count). `num_threads=4` and the seed are fixed so the run is
reproducible.

In [7]:
PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=127, min_data_in_leaf=500,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              max_bin=255, num_threads=4, verbose=-1, seed=0)
ROUNDS = 600
CATEGORICAL = ["s_unit", "s_plant"]

y = META["y"].to_numpy()
train = ~np.isnan(y)
t0 = time.time()
ds = lgb.Dataset(X[train], label=y[train], feature_name=NAMES,
                 categorical_feature=CATEGORICAL, free_raw_data=True)
booster = lgb.train(PARAMS, ds, num_boost_round=ROUNDS)
print(f"trained on {train.sum():,} rows in {time.time() - t0:.0f}s")

pred = META[~train].copy()
pred["p"] = booster.predict(X[~train])

trained on 6,872,544 rows in 303s


In [8]:
# wide submission: one row per test case, columns in the sample submission's order
units = list(TOPO.units)
sample = pd.read_csv(ROOT / "sample_submission.csv")
wide = pred.pivot_table(index="run_no", columns=["unit", "hour"], values="p")
cols = {}
for c in sample.columns[1:]:
    g = re.fullmatch(r"result_committed_(.+)_t(\d+)", c)
    cols[c] = wide[(units.index(g.group(1)), int(g.group(2)))].to_numpy()
submission = pd.DataFrame(cols, index=wide.index).rename_axis("Run No").reset_index()

assert list(submission.columns) == list(sample.columns)
assert set(submission["Run No"]) == set(sample["Run No"]) and len(submission) == len(sample)
assert submission.drop(columns="Run No").isna().sum().sum() == 0
submission.to_csv(OUT_PATH, index=False)
print(f"wrote {OUT_PATH} {submission.shape}")

wrote submission_lp_lightgbm.csv (725, 2353)


In [9]:
# Optional: compare with the file we uploaded to Kaggle, if it is next to this notebook.
ref_path = ROOT / "submission_physics.csv"
if ref_path.exists():
    ref = pd.read_csv(ref_path).set_index("Run No").loc[submission["Run No"]]
    diff = np.abs(submission.drop(columns="Run No").to_numpy() - ref.to_numpy())
    print(f"max |difference| to the uploaded submission: {diff.max():.2e}")

max |difference| to the uploaded submission: 2.19e-13
